# ISLES'22 - Rapor paketi

Final rapor yerelde uretiliyor (`rapor/build_report.py`), ama figurlerin ve ozet tablolarin
cogu Drive'da. Bu notebook onlari tek bir zip'te topluyor.

Toplanan dosyalar rapora giren malzeme: veri kesfi ve on isleme figurleri, iki aday modelin
egitim egrileri ve vaka gorselleri, degerlendirme ciktilari (esik taramasi, forest plot,
bootstrap tablosu) ve karsilastirma tablosu.

Bastan sona calistir, sonra `rapor_paketi.zip` dosyasini indirip yerelde
`iskemik_inme/rapor/girdi/` altina ac.

In [1]:
import os, json, shutil
from pathlib import Path

ORTAM = 'colab' if os.path.isdir('/content') else 'yerel'


def drive_bagla(nokta='/content/drive'):
    if ORTAM != 'colab':
        return None
    from google.colab import drive
    if os.path.ismount(nokta):
        print('Drive zaten bagli:', nokta)
        return nokta
    try:
        drive.mount(nokta)
    except ValueError as e:
        print('duz mount basarisiz (%s) -> force_remount' % e)
        drive.mount(nokta, force_remount=True)
    return nokta


drive_bagla()
PROJE = '/content/drive/MyDrive/iskemik_inme' if ORTAM == 'colab' else str(Path.cwd())
PAKET = os.path.join(PROJE, 'rapor_paketi')
shutil.rmtree(PAKET, ignore_errors=True)
os.makedirs(PAKET, exist_ok=True)
print('PROJE:', PROJE)

Drive zaten bagli: /content/drive
PROJE: /content/drive/MyDrive/iskemik_inme


In [2]:
ADAYLAR = ['unet3d', 'unet_r34_2d']
TUM_MODELLER = ['unet_r34_2d', 'unet_r34_25d', 'segformer_b0', 'unet_mbv3', 'unet3d',
                'A1_flair', 'A2_tversky', 'A3_k1', 'A3_k3']

ISTENEN = []
# veri ve on isleme figurleri
for f in ('veri_kesfi.png', 'ornek_vakalar.png', 'adc_olcek_duzeltme.png',
          'onisleme_dogrulama.png', 'flair_hizalama.png'):
    ISTENEN.append(('ozet/' + f, f))
# ozet tablolari
for f in ('kesif_ozeti.json', 'hazirlik_ozeti.json', 'flair_ozeti.json',
          'flair_hizalama.xlsx', 'haric_vakalar.json'):
    ISTENEN.append(('ozet/' + f, f))
# degerlendirme
for f in ('esik_taramasi.png', 'forest_dice.png', 'bootstrap_ci.xlsx',
          'esik_bilesen_etkisi.xlsx', 'secim.json', 'degerlendirme_ozeti.json',
          'panoptica_karsilastirma.xlsx'):
    ISTENEN.append(('degerlendirme/' + f, f))
for mk in ADAYLAR:
    ISTENEN.append((f'degerlendirme/{mk}_test_secimli.xlsx', f'{mk}_test_secimli.xlsx'))
# karsilastirma
ISTENEN.append(('karsilastirma_tablosu.xlsx', 'karsilastirma_tablosu.xlsx'))
# egitim egrileri: tum modeller (kucuk dosyalar)
for mk in TUM_MODELLER:
    ISTENEN.append((f'{mk}/{mk}_curves.png', f'{mk}_curves.png'))
    ISTENEN.append((f'{mk}/{mk}_log.xlsx', f'{mk}_log.xlsx'))
# vaka gorselleri: yalniz iki aday
for mk in ADAYLAR:
    ISTENEN.append((f'{mk}/{mk}_cases/{mk}_cases.png', f'{mk}_cases.png'))

bulunan, eksik = [], []
for kaynak, hedef in ISTENEN:
    y = os.path.join(PROJE, kaynak)
    if os.path.exists(y):
        shutil.copy2(y, os.path.join(PAKET, hedef))
        bulunan.append(hedef)
    else:
        eksik.append(kaynak)

print('kopyalanan:', len(bulunan))
if eksik:
    print()
    print('BULUNAMAYAN', len(eksik), 'dosya:')
    for q in eksik:
        print('  ', q)

kopyalanan: 40


In [3]:
zip_yol = shutil.make_archive(os.path.join(PROJE, 'rapor_paketi'), 'zip', PAKET)
mb = os.path.getsize(zip_yol) / 1e6
print('ZIP HAZIR:', zip_yol, round(mb, 1), 'MB')
print()
for q in sorted(os.listdir(PAKET)):
    print('  ', q, round(os.path.getsize(os.path.join(PAKET, q)) / 1e6, 2), 'MB')
print()
print('INDIRME:')
print('  1) drive.google.com -> Drive"im -> iskemik_inme')
print('  2) rapor_paketi.zip -> sag tik -> Indir')
print('  3) Zipi ac -> d:/mamografi/iskemik_inme/rapor/girdi/ altina koy')
print('  4) Yerelde: python rapor/build_report.py')

ZIP HAZIR: /content/drive/MyDrive/iskemik_inme/rapor_paketi.zip 2.7 MB

   A1_flair_curves.png 0.2 MB
   A1_flair_log.xlsx 0.01 MB
   A2_tversky_curves.png 0.2 MB
   A2_tversky_log.xlsx 0.02 MB
   A3_k1_curves.png 0.21 MB
   A3_k1_log.xlsx 0.01 MB
   A3_k3_curves.png 0.2 MB
   A3_k3_log.xlsx 0.01 MB
   adc_olcek_duzeltme.png 0.04 MB
   bootstrap_ci.xlsx 0.01 MB
   degerlendirme_ozeti.json 0.0 MB
   esik_bilesen_etkisi.xlsx 0.01 MB
   esik_taramasi.png 0.11 MB
   flair_hizalama.png 0.15 MB
   flair_hizalama.xlsx 0.01 MB
   flair_ozeti.json 0.0 MB
   forest_dice.png 0.05 MB
   haric_vakalar.json 0.0 MB
   hazirlik_ozeti.json 0.0 MB
   karsilastirma_tablosu.xlsx 0.01 MB
   kesif_ozeti.json 0.0 MB
   onisleme_dogrulama.png 0.12 MB
   ornek_vakalar.png 0.14 MB
   panoptica_karsilastirma.xlsx 0.01 MB
   secim.json 0.0 MB
   segformer_b0_curves.png 0.22 MB
   segformer_b0_log.xlsx 0.01 MB
   unet3d_cases.png 0.15 MB
   unet3d_curves.png 0.22 MB
   unet3d_log.xlsx 0.01 MB
   unet3d_test_seciml